# Infrared Image Colorization and Enhancement Framework
### Satellite and Thermal Remote Sensing Image Enhancement & Translation (FLIR Thermal Dataset)

This notebook provides the complete pipeline to train the **Attention U-Net cGAN (Pix2Pix)** model on the FLIR Thermal dataset using GPU acceleration in Google Colab, verify its colorization accuracy, and run YOLOv8n object detection comparison.

## Step 1: Environment Setup & Google Drive Mount
Mount your Google Drive to save trained model weights persistently so that you can download them later for the web dashboard.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check if GPU is active
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Step 2: Install Dependencies

In [ ]:
!pip install ultralytics opencv-python-headless scikit-image pillow matplotlib numpy

## Step 3: Kaggle API & FLIR Dataset Download
1. Go to your Kaggle Account page.
2. Click **Create New API Token** to download `kaggle.json`.
3. Upload `kaggle.json` using the file explorer sidebar on the left.

In [ ]:
# Setup Kaggle API credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the FLIR Thermal Dataset (10k paired images)
# Replace with the exact dataset URL identifier
!kaggle datasets download -d deepquestai/flir-thermal-dataset
!unzip -q flir-thermal-dataset.zip -d /content/flir_dataset

## Step 4: Write Model Files Locally in Colab
Let's write the core files: `generator.py`, `discriminator.py`, `dataset.py`, and `train.py`.

In [ ]:
# Make workspace directory structure
!mkdir -p models utils

print("Ready to import models and train. Please upload your training files or create them using file manager.")

## Step 5: Start Model Training
Run the training script for 50-100 epochs using the Adam Optimizer, L1 + GAN loss.

In [ ]:
# Execute training on the FLIR dataset
!python train.py \
    --dataset_path /content/flir_dataset \
    --checkpoint_dir /content/models \
    --epochs 50 \
    --batch_size 16 \
    --lr 0.0002 \
    --lambda_L1 100.0

## Step 6: Backup Checkpoints to Google Drive

In [ ]:
!cp /content/models/generator.pth "/content/drive/MyDrive/generator.pth"
!cp /content/models/discriminator.pth "/content/drive/MyDrive/discriminator.pth"
print("Weights successfully copied to your Google Drive!")

## Step 7: Evaluate Model & Run Object Detection

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from models.generator import AttentionUNetGenerator
from utils.enhancement import enhance_ir_image
from utils.metrics import calculate_psnr, calculate_ssim
from models.yolov8_helper import ObjectDetector

# 1. Load Generator model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = AttentionUNetGenerator(in_channels=1, out_channels=3).to(device)
generator.load_state_dict(torch.load("/content/models/generator.pth", map_location=device))
generator.eval()

# 2. Load Object Detector
detector = ObjectDetector()

# 3. Test on a sample test image
# Load a test image (change path to real path)
test_ir_path = "/content/flir_dataset/test/thermal/mock_image_0001.png" # change to actual
test_gt_path = "/content/flir_dataset/test/rgb/mock_image_0001.png" # change to actual

try:
    ir_gray = cv2.imread(test_ir_path, cv2.IMREAD_GRAYSCALE)
    gt_bgr = cv2.imread(test_gt_path)
    
    # Preprocess / Enhance
    enhanced = enhance_ir_image(ir_gray)
    
    # Inference
    resized = cv2.resize(enhanced, (256, 256))
    tensor_in = torch.tensor(resized, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device) / 127.5 - 1.0
    with torch.no_grad():
        tensor_out = generator(tensor_in)
    
    out_rgb = (tensor_out.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) * 127.5
    out_rgb = np.clip(out_rgb, 0, 255).astype(np.uint8)
    out_bgr = cv2.cvtColor(out_rgb, cv2.COLOR_RGB2BGR)
    out_bgr_resized = cv2.resize(out_bgr, (ir_gray.shape[1], ir_gray.shape[0]))
    
    # Run Object Detection
    ir_3ch = cv2.cvtColor(ir_gray, cv2.COLOR_GRAY2BGR)
    dets_ir = detector.run_detection(ir_3ch, is_colorized=False)
    dets_rgb = detector.run_detection(out_bgr_resized, is_colorized=True)
    
    annotated_ir = detector.draw_detections(ir_3ch, dets_ir)
    annotated_rgb = detector.draw_detections(out_bgr_resized, dets_rgb)
    
    # Calculate Metrics
    psnr_score = calculate_psnr(gt_bgr, out_bgr_resized)
    ssim_score = calculate_ssim(gt_bgr, out_bgr_resized)
    
    # Plot Results
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(ir_gray, cmap='gray')
    axes[0, 0].set_title("Original Infrared (Input)")
    
    axes[0, 1].imshow(enhanced, cmap='gray')
    axes[0, 1].set_title("Enhanced IR (CLAHE + Sharp)")
    
    axes[0, 2].imshow(cv2.cvtColor(out_bgr_resized, cv2.COLOR_BGR2RGB))
    axes[0, 2].set_title(f"Colorized RGB (PSNR: {psnr_score:.2f}dB, SSIM: {ssim_score:.3f})")
    
    axes[1, 0].imshow(cv2.cvtColor(annotated_ir, cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title(f"Detections on Original IR ({len(dets_ir)} items)")
    
    axes[1, 1].imshow(cv2.cvtColor(annotated_rgb, cv2.COLOR_BGR2RGB))
    axes[1, 1].set_title(f"Detections on Colorized RGB ({len(dets_rgb)} items)")
    
    axes[1, 2].imshow(cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title("Ground Truth RGB")
    
    for row in axes:
        for ax in row:
            ax.axis('off')
            
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Make sure to replace test_ir_path and test_gt_path with real image paths: {e}")